# POLITE — 2026-09-07 observation

POLITE's first pointed night. The whole program is four reviewed
YAML plans; this notebook launches them, watches frames land, and inspects what
arrived. It never opens the camera itself.

**Goal:** a usable V-band flat field and a four-star polarimetric calibration, plus
the 50-bias set that closes Mode 5 / gain 56 / offset 20 read noise in electrons.
**Not tonight:** the even-illumination panel (not installed), science targets,
Mode 3 read noise (deferred — it needs its own bias set and its own PTC light
source), reduction-pipeline tweaks.

Operator sheet with the sky table, timeline, and target windows:
`night_plans/20260907_night_sheet.md`.

## Operating rules

- **One acquisition path.** `scripts/execute_night.py <plan> --run` owns the camera,
  EFW, HWP, and mount for the duration of a block. Do not run a capture cell from
  `notebooks/templates/capture.ipynb` while a plan is running.
- **Every `live.*` cell is read-only** — it reads FITS off disk and touches no
  hardware, so it is safe to run while a plan is in flight.
- **Cells that command motion are commented out.** Uncomment deliberately, one at a
  time, after reading the cell above it.
- **`FITSDATA/` is never modified.** Each invocation writes into
  `FITSDATA/20260907/<subdir>/`. Give a repeated block a *new* subdir.
- **PWI4 `alt` is conventional altitude** (90 = zenith, 0 = horizon). The shed-safe
  window is altitude **42–90°** — and **60–90° in the NE quadrant (az 0–90°)**, where
  the shed wall is higher. Checked after the slew by the runner; a violation aborts
  that invocation.

## TONIGHT AS ACTUALLY RUN — revised 20:00 PDT

**Twilight flats did not happen.** Nautical twilight passed at 19:57 while the Alpaca
servers were down and the runner's HWP connect was broken. §4 below does **not** run.

**Consequence for reduction:** no flat means the `lsq` default is compromised. Reduce
with **`double_ratio`**, the documented bad-flat-night fallback — dual-beam double ratio
cancels flat-field response and transparency to first order. The standards are still good.

**Order tonight — clear sky is the perishable resource, darks are not:**

| When | What | Why |
|---|---|---|
| 20:00 | **standards1** — run now | ~1 h of clear sky; 9–11 min per target |
| when sky closes | **darkcal** — and it parks | cloud- and time-immune, do it last |

Twenty minutes gets the whole first-order calibration: HD 154345 (unpolarized → zero
point) then HD 183143 (polarized → PA zero + modulation efficiency). Targets after that
are confirmation. Plan order already front-loads them, so it runs unmodified.

```
python scripts/execute_night.py night_plans/20260907_standards.yaml --run \
  --subdir standards1 --yes --no-mount-home

python scripts/execute_night.py night_plans/20260907_darkcal.yaml --run \
  --mount on --park-on-finish
```

An aborted run never parks. If you Ctrl-C the standards, still run darkcal (or park by hand).

**Convention settled 2026-09-07:** PWI4 altitude is conventional — **90 = zenith**, and the
shed-safe window is **altitude 42–90°**. Every earlier 'zenith distance 3–42' statement is dead.


## 0 · Shared preamble

Run these two cells first. They are byte-identical to the template family.

In [1]:
import os, sys
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root: _root = _root.parent
if not (_root / 'pyproject.toml').exists(): raise RuntimeError('Could not locate the POLITE repository root')
os.chdir(_root)
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
print('POLITE root:', _root)


POLITE root: C:\Users\ASTRO\Desktop\POLITE


In [2]:
from pathlib import Path
from obs_utils import live
from obs_utils import interactive as obs
from obs_utils import user_config as uc
SESSION_DIR = None  # set in Tonight's card


## 1 · Tonight's card

Provenance, not a camera override: the detector operating point lives in each plan's
`camera:` block (Mode 5 / gain 56 / offset 20, cooler −10 °C) and the runner reads it
back off the hardware before the first frame.

In [3]:
import subprocess, shutil

NIGHT = '20260907'
PROGRAM = 'standards + twilight flats + Mode 5 read noise'
SESSION_DIR = Path('FITSDATA') / NIGHT
SESSION_DIR.mkdir(parents=True, exist_ok=True)

PYTHON = Path('/Users/blu3/miniforge3/envs/POLITE/bin/python')
if not PYTHON.exists(): PYTHON = Path(sys.executable)

PLANS = Path('night_plans')
FLATS      = PLANS / '20260907_twilight_flats.yaml'
STANDARDS  = PLANS / '20260907_standards.yaml'
STANDARDS2 = PLANS / '20260907_standards_pass2.yaml'
DARKCAL    = PLANS / '20260907_darkcal.yaml'

# subdir -> the flags the night sheet specifies for that invocation
BLOCKS = {
    'twiflat':    (FLATS,      ['--no-mount-home']),
    'standards1': (STANDARDS,  ['--no-mount-home']),
    'standards2': (STANDARDS2, ['--no-mount-home']),
    'darkcal':    (DARKCAL,    ['--mount', 'on', '--no-mount-home', '--park-on-finish']),
}

print('session:', SESSION_DIR, '| program:', PROGRAM)
print('python :', PYTHON)

session: FITSDATA\20260907 | program: standards + twilight flats + Mode 5 read noise
python : c:\Users\ASTRO\Desktop\POLITE\.venv\Scripts\python.exe


In [4]:
def runner_command(plan, *, subdir=None, run=False, extra=()):
    """The exact argv for one execute_night invocation. No --setpoint: every plan
    carries its own `camera:` block and the runner reads it from there."""
    cmd = [str(PYTHON), 'scripts/execute_night.py', str(plan)]
    if run:
        cmd += ['--run', '--yes']
    if subdir is not None:
        cmd += ['--subdir', subdir]
    return cmd + list(extra)


def block_dir(subdir):
    """Where a given --subdir writes. Explicit --subdir means no _HHMM suffix."""
    return SESSION_DIR / subdir


def preview(subdir):
    """Read-only dry-run. Touches no hardware."""
    plan, extra = BLOCKS[subdir]
    subprocess.run(runner_command(plan, subdir=subdir, extra=extra), cwd=Path.cwd(), check=True)


def launch(subdir):
    """MOTION. Start the runner in the BACKGROUND so the live cells below can watch
    frames land. Returns the Popen handle; its stdout is teed to a log file.

    Interrupting the kernel does NOT stop the child -- use proc.terminate(), or run
    the block from a terminal instead (that is the authoritative path for the
    unattended chain)."""
    plan, extra = BLOCKS[subdir]
    cmd = runner_command(plan, subdir=subdir, run=True, extra=extra)
    logs = SESSION_DIR / 'logs'; logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / f'notebook_{subdir}.log'
    print('MOTION -- runner owns camera/EFW/HWP/mount:')
    print(' ', ' '.join(cmd))
    print('  log:', log_path)
    handle = open(log_path, 'w')
    return subprocess.Popen(cmd, cwd=Path.cwd(), stdout=handle, stderr=subprocess.STDOUT, text=True)


def tail(subdir, n=40):
    """Last n lines of a launched block's runner output."""
    log_path = SESSION_DIR / 'logs' / f'notebook_{subdir}.log'
    if not log_path.exists():
        print('no log yet:', log_path); return
    print(''.join(log_path.read_text(errors='replace').splitlines(keepends=True)[-n:]))

In [5]:
# Pre-flight: the plans exist, and there is room for 811 frames x 52.9 MB ~= 43 GB.
for subdir, (plan, extra) in BLOCKS.items():
    print(f"{subdir:<11s} {plan}  {'OK' if plan.exists() else 'MISSING'}  {' '.join(extra)}")

free_gb = shutil.disk_usage(SESSION_DIR).free / 1024**3
print(f'\nfree on the data volume: {free_gb:.1f} GB  (need >= 50 GB)')
if free_gb < 50:
    print('NOT ENOUGH ROOM -- trim pass 2 (-11 GB) or the 0.3 s rungs (-7 GB), '
          'or free space before 19:05.')

twiflat     night_plans\20260907_twilight_flats.yaml  OK  --no-mount-home
standards1  night_plans\20260907_standards.yaml  OK  --no-mount-home
standards2  night_plans\20260907_standards_pass2.yaml  OK  --no-mount-home
darkcal     night_plans\20260907_darkcal.yaml  OK  --mount on --no-mount-home --park-on-finish

free on the data volume: 688.1 GB  (need >= 50 GB)


## 2 · Before dark — bring-up and fail-closed gates (18:00, attended)

Connect, verify, cool. Nothing here slews the mount: **home the mount once by hand in
PWI4**, then every plan below runs with `--no-mount-home` so no unattended block
re-homes the mount mid-night.

In [20]:
# RUN THIS FIRST -- obs.connect_all() only CONNECTS; it starts no server, so with the
# servers down camera, filter wheel and HWP all fail together. Idempotent: each server
# is launched only if its own management endpoint is not already answering.
#   ASCOM Remote :11111 -> ZWO EFW + Optec Pyxis HWP
#   QHY Alpaca   :11112 -> QHY268M camera   (close EZCAP first, USB is exclusive)
# Observatory Windows PC only; skip on the lab Mac.
from obs_utils.alpaca_servers import start_observatory_alpaca_servers
start_observatory_alpaca_servers(
    ascom_endpoint=uc.ALPACA_CONFIG.host,
    qhy_endpoint=uc.ALPACA_CONFIG.camera_host,
)

ASCOM Remote already ready at localhost:11111
QHY Alpaca already ready at localhost:11112
Alpaca servers ready: ASCOM Remote (localhost:11111); QHY Alpaca (localhost:11112)


AlpacaServerStatus(ascom_endpoint='localhost:11111', qhy_endpoint='localhost:11112', ascom_started=False, qhy_started=False)

In [6]:
s = obs.connect_all()
s.status()

connected : camera, filter wheel, hwp (alpaca), mount (pwi4 client), focuser, field rotator
POLITE interactive session
----------------------------------------------------
camera     : connected  (6280x4210)
filter     : slot 2  (Photometric V)
hwp        : alpaca  pos=0.000 deg  moving=False
mount      : connected=True slewing=False tracking=True  ra=21.2659h dec=25.434d
focuser    : connected=True enabled=True pos=28405.0 moving=False
field rot  : connected=True enabled=True field=-0.008 deg moving=False


{'instrument': {'connected': True,
  'camera_connected': True,
  'sensor': (6280, 4210),
  'filter': {'slot': 2, 'name': 'Photometric V'}},
 'hwp': {'backend': 'alpaca', 'position_deg': 0.0, 'moving': False},
 'mount': {'connected': True,
  'slewing': False,
  'tracking': True,
  'ra_hours': 21.2659457025729,
  'dec_deg': 25.4344237601866},
 'focuser': {'exists': True,
  'connected': True,
  'enabled': True,
  'position': 28404.9920883023,
  'moving': False},
 'field_rotator': {'exists': True,
  'connected': True,
  'enabled': True,
  'mech_deg': 0.00705888815644913,
  'field_deg': -0.00844828637020681,
  'moving': False}}

In [8]:
from obs_utils.night_safety import INSTALLED_EFW_NAMES, verify_filter_wheel
verify_filter_wheel(s.imaging)
print(INSTALLED_EFW_NAMES)

['Clear', 'Photometric B', 'Photometric V', 'Photometric R', 'Dark']


In [9]:
# The cal blocks are shutterless darks: slot 5 ('Dark') is the only light block the
# QHY268M has. Confirm the driver actually reports it before trusting a BIAS frame.
from obs_utils.night_safety import cooler_gate
cam = s.camera
print('mode/gain/offset now:', cam.ReadoutMode, cam.Gain, cam.Offset)
cam.SetCCDTemperature = -10.0
cooler_gate(cam, -10.0, tol_c=0.5, stable_s=30.0, timeout_s=900.0)

mode/gain/offset now: 5 56 20
[cooler] waiting for -10.0 C (tol 0.50 C, hold 30 s, timeout 900 s)
[cooler] T= -9.70 C  target=-10.0 C  power=67%  elapsed=   0 s
[cooler] T= -9.00 C  target=-10.0 C  power=76%  elapsed=   5 s
[cooler] T= -9.20 C  target=-10.0 C  power=80%  elapsed=  10 s
[cooler] T= -9.50 C  target=-10.0 C  power=80%  elapsed=  15 s
[cooler] T= -9.60 C  target=-10.0 C  power=79%  elapsed=  20 s
[cooler] T= -9.90 C  target=-10.0 C  power=78%  elapsed=  25 s
[cooler] T=-10.00 C  target=-10.0 C  power=78%  elapsed=  30 s
[cooler] T=-10.10 C  target=-10.0 C  power=76%  elapsed=  35 s
[cooler] T=-10.20 C  target=-10.0 C  power=75%  elapsed=  40 s
[cooler] T=-10.30 C  target=-10.0 C  power=73%  elapsed=  45 s
[cooler] STABLE at -10.30 C (power 73%) after 45 s


-10.3

In [10]:
# MOTION -- home the HWP once per power cycle, on the serial path.
# s = obs.connect_hwp_serial()
# s.home_hwp()

In [11]:
# MOTION -- prove the stage responds and lands inside tolerance before science.
# from obs_utils.night_safety import HWP_DEFAULT_TOL_DEG
# achieved = s.hwp(22.5)
# print('commanded 22.5, achieved', achieved)
# assert abs(achieved - 22.5) <= HWP_DEFAULT_TOL_DEG
# s.hwp(0.0)

### Mount check — by hand, in PWI4

Enable both axes and home **once, in PWI4 itself**. Every plan below then runs with
`--no-mount-home`, so no unattended block re-homes the mount mid-night. The cell below
is query-only — it reads the current PWI4 altitude and says whether the mount is
inside the shed-safe window right now (being outside is normal when parked). The runner
repeats this check *after* its own slew.

In [12]:
# Query only -- no connect, home, slew, or capture.
from obs_utils.config import default_sky_regions
from obs_utils.obs_math import airmass_kasten_young
from obs_utils.mount import _altaz_allowed
from obs_utils.pwi4_client import PWI4
from obs_utils.user_config import PWI4_CONFIG

pwi4 = PWI4(host=PWI4_CONFIG.host, port=PWI4_CONFIG.port)
status = pwi4.status()
# PWI4 reports conventional altitude directly: 90 = zenith, 0 = horizon. Do NOT
# run it through zenith_distance_to_altitude -- that would compute 90 - alt.
alt = float(status.mount.altitude_degs)
az = float(status.mount.azimuth_degs)
regions = default_sky_regions()   # azimuth-dependent since 2026-09-07; a list

print(f'PWI4 alt/az   : {alt:.2f} / {az:.2f} deg  (90 = zenith)')
print(f'airmass       : {airmass_kasten_young(alt)}')
print('allowed regions:')
for r in regions:
    print(f'  {r.name:40s} alt {r.alt_min_deg:.0f}--{r.alt_max_deg:.0f}, '
          f'az {r.az_min_deg:.0f}--{r.az_max_deg:.0f} deg')
print('axes enabled  :', status.mount.axis0.is_enabled, status.mount.axis1.is_enabled)
# Azimuth matters: the NE quadrant floor is 60 deg, elsewhere 42. Use the same
# check the runner enforces after its slew.
print('inside window' if _altaz_allowed(alt, az, regions)
      else 'outside window (normal when parked; the runner checks after its slew)')


PWI4 alt/az   : 72.64 / 110.97 deg  (90 = zenith)
airmass       : 1.0472802329014923
allowed regions:
  pwi4_ne_shed_wall_alt_60_to_90_deg       alt 60--90, az 0--90 deg
  pwi4_altitude_42_to_90_deg               alt 42--90, az 90--360 deg
axes enabled  : True True
inside window


## 3 · Dry-run every plan (18:30)

Read-only. Confirm frame counts, HWP angles, targets, the −10 °C setpoint read from
each plan's `camera:` block, and the predicted mount actions. Expected totals:
**twiflat 224 · standards1 291 · standards2 216 · darkcal 80 = 811 frames**.

In [13]:
for subdir in BLOCKS:
    print('=' * 72); print(subdir); print('=' * 72)
    preview(subdir)

twiflat
standards1
standards2


KeyboardInterrupt: 

## 6 · Standards + calibration — 20:30, unattended

**The authoritative path is a terminal**, so the chain survives a closed laptop or a
dead kernel. Three invocations joined with `;` — never `&&` — so an abort in one does
not cancel the rest:

```zsh
scripts/execute_night.py night_plans/20260907_standards.yaml \
  --run --subdir standards1 --yes --no-mount-home ; \
scripts/execute_night.py night_plans/20260907_standards_pass2.yaml \
  --run --subdir standards2 --yes --no-mount-home ; \
scripts/execute_night.py night_plans/20260907_darkcal.yaml \
  --run --subdir darkcal --yes --mount on --no-mount-home --park-on-finish
```

**Parking rides on `darkcal`, on purpose.** `--park-on-finish` fires only after a clean
finish (`scripts/execute_night.py:441` sits outside the run block, and an abort
propagates past it). `darkcal` commands no slews, so it is the one invocation that
cannot abort on the altitude gate — which makes it the reliable parker even if
both science plans die. Do not reorder the chain.

HD 154345 is first because its window closes at 22:12; it is deliberately absent from
pass 2, where a 22:12+ slew would abort the run. Pass 2 is optional — dropping it costs
nothing but the second-epoch parallactic-angle diagnostic.

In [ ]:
preview('standards1'); preview('standards2'); preview('darkcal')

In [ ]:
# MOTION -- notebook equivalent of the terminal chain. Use the terminal instead unless
# you are staying with the kernel; `;` semantics are reproduced by NOT raising between
# blocks, so an aborted plan does not cancel the ones after it.
# for subdir in ('standards1', 'standards2', 'darkcal'):
#     proc = launch(subdir)
#     rc = proc.wait()
#     print(f'{subdir}: exit {rc}' + ('' if rc == 0 else '  <-- ABORTED, continuing'))

In [10]:
# Read-only, safe while the chain runs. One figure per complete 8-angle cycle.
science_stats = live.watch(block_dir('standards1'), timeout_s=3600, every=8)

watch interrupted


In [11]:
science_stats = live.watch(block_dir('standards2'), timeout_s=3600, every=8)

watch interrupted


In [ ]:
tail('standards1', 40)

## 7 · Read-only checks after each block

Provenance and completeness only. The instrumental q/u estimate and the
polarized-standard P/PA comparison need tracked paired-aperture photometry and belong
in the dated reduction notebook — do not infer them from whole-frame statistics here.

In [ ]:
for subdir in BLOCKS:
    d = block_dir(subdir)
    n = len(live.find_frames(d))
    print('=' * 72); print(f'{subdir}  ({n} frames)  {d}'); print('=' * 72)
    if n:
        live.session_table(d)

In [ ]:
live.hwp_coverage(block_dir('standards1'))
live.qa_print(live.sequence_audit(block_dir('standards1')))

In [ ]:
# Clipping is the one thing that voids a target's complete HWP cycle. Measure once,
# then group. This reads 291 full frames, so give it a minute.
lights = live.stats_table(live.select(block_dir('standards1'), imagetyp='LIGHT'), show=False)
live.group_table(lights, by=('object_name', 'exptime'))
voided = {(st.object_name, st.exptime) for st in lights if st.saturated_px}
for st in lights:
    if st.saturated_px:
        print('SATURATED --', st.line())
print('\ntarget/exposure cycles voided by clipping:', sorted(voided) or 'none')

In [ ]:
# Modulation should show as a smooth run of median with HWP angle for a polarized
# standard and a flat line for an unpolarized one. Indicative only, NOT a measurement:
# whole-frame medians are not paired-aperture photometry. The real q/u belongs in the
# dated reduction notebook.
live.trend([st for st in lights if st.object_name == 'HD 183143'],
           x='hwp_angle_deg', y='median', by='exptime')

### The Mode 5 read-noise set

`darkcal` carries 50 bias frames at Mode 5 / gain 56 / offset 20. `bias_qa` is the
gate: level and stability, sigma-clipped. The **measurement** — σ of (biasᵢ − biasⱼ)/√2
over many pairs, the Janesick difference-pair method — belongs in the reduction
notebook via `caltools`, not here.

Offset 20 puts the pedestal at roughly 50–100 ADU, so the bias distribution's left tail
is intact and the measured σ is the real read noise. Worth a glance at the histogram;
it is not a gate.

In [ ]:
bias_paths = live.select(block_dir('darkcal'), imagetyp='BIAS')
print(len(bias_paths), 'bias frames (expect 50)')
live.qa_print(live.bias_qa(bias_paths))
live.histogram(bias_paths)   # left tail should clear zero with room to spare

In [ ]:
darks = live.stats_table(live.select(block_dir('darkcal'), imagetyp='DARK'), show=False)
live.group_table(darks, by=('exptime',))
live.trend(darks, x='exptime', y='median')     # slope -> dark rate at Mode 5, -10 C
live.trend(darks, x='time', y='det_temp_c')    # cooler held? drift voids the slope

## 8 · PlateSolve3 — optional, after the data is in

Nothing above depends on this and it must not delay a block or the closeout. It is a
**read-only commissioning probe on frames already written to disk**: record PWI4 state
before and after plus the raw PS3 output fields, and make no mount or pointing-model
change tonight. The solver has never been run on a Savart-doubled field — every star
appears twice — so inspect the gray display before trusting anything it reports.

Skip it entirely if the night ran late, or if the PS3CLI executable and Kepler
catalogue are not installed on this machine. A live solve during acquisition is the
eventual goal and is not tonight's business; tonight only asks whether PS3 returns
anything sane on a POLITE frame.

In [ ]:
# Prefer a real captured standards frame -- the point of solving after the fact is that
# the data is already on disk. Falls back to the attended probe frame if one was taken.
_solve_candidates = (live.find_frames(block_dir('standards1'))
                     or live.find_frames(block_dir('standards2')))
PLATE_FRAME = (Path(_solve_candidates[0]) if _solve_candidates
               else SESSION_DIR / 'pointcheck_probe.fits')
PLATE_SCALE_ARCSEC_PER_PX = 0.224
PS3CLI_EXE = Path(r'C:\SET\PS3CLI\ps3cli.exe')   # set on the observatory PC
PS3_CATALOG = Path(r'C:\SET\Kepler')              # set on the observatory PC
print('frame to solve:', PLATE_FRAME)
if PLATE_FRAME.exists():
    live.show_frame(PLATE_FRAME, cmap='gray', percentile_clip=(5, 99.8))
else:
    print('nothing on disk to solve -- skip this section')

In [ ]:
# Query-only: no slew, no offset, no capture, no model update, no change to the FITS.
# from obs_utils.platesolve import PlateSolveConfig, platesolve
# if s.pwi4 is None: raise RuntimeError('Run obs.connect_all() first; this is query-only.')
# pwi_before = s.pwi4.status()
# plate_result = platesolve(PLATE_FRAME, PLATE_SCALE_ARCSEC_PER_PX, PlateSolveConfig(PS3CLI_EXE, PS3_CATALOG))
# pwi_after = s.pwi4.status()
# print('PWI4 J2000 before/after:',
#       (pwi_before.mount.ra_j2000_hours, pwi_before.mount.dec_j2000_degs),
#       (pwi_after.mount.ra_j2000_hours, pwi_after.mount.dec_j2000_degs))
# print('PS3 raw fields:', dict(plate_result.raw_fields))

## Closeout

The runner closes the camera, EFW, and HWP itself after each block, and `darkcal` parks
the mount. Preserve the raw FITS, `block_manifest.jsonl`, and `pol_config.yaml` from
every subdirectory, plus `FITSDATA/20260907/logs/`.

Record in the observing log: which blocks completed, the usable flat rungs, any
clipping (and which target cycle it voided), whether pass 2 ran, the post-slew PWI4
altitude for each target, whether the PlateSolve3 proof was attempted and what
raw fields it returned, and whether the mount parked.

**Every number in the plan files was a prediction until tonight.** Say plainly in the
log which ones survived contact.

In [7]:
for subdir in BLOCKS:
    d = block_dir(subdir)
    if live.find_frames(d):
        print('=' * 72); print(subdir); print('=' * 72)
        live.session_table(d)
        live.qa_print(live.sequence_audit(d))

standards1
FITSDATA\20260907\standards1: 291 frame(s)
  LIGHT     0.300s  Photometric V    x96
  LIGHT     1.500s  Photometric V    x96
  LIGHT     2.000s  Clear            x3
  LIGHT     6.000s  Photometric V    x96
[PASS] sequence_audit
  PASS: 12 science sequence(s) complete
standards2
FITSDATA\20260907\standards2: 216 frame(s)
  LIGHT     0.300s  Photometric V    x72
  LIGHT     1.500s  Photometric V    x72
  LIGHT     6.000s  Photometric V    x72
[PASS] sequence_audit
  PASS: 9 science sequence(s) complete
darkcal
FITSDATA\20260907\darkcal: 80 frame(s)
  BIAS      0.000s  Dark             x50
  DARK      6.000s  Dark             x10
  DARK     20.000s  Dark             x10
  DARK     60.000s  Dark             x10
[PASS] sequence_audit
  PASS: 0 science sequence(s) complete


In [8]:
# Read-only. Whole-night trends across every block that produced frames: per-frame
# detector temperature (reduction uses this, not the setpoint) and level/noise drift.
import matplotlib.pyplot as plt

all_stats = [live.frame_stats(f)
             for subdir in BLOCKS
             for f in live.find_frames(block_dir(subdir))]
print(f'{len(all_stats)} frames across {len(BLOCKS)} blocks')
if all_stats:
    live.temperature_trend(all_stats); plt.show()
    live.level_trend(all_stats); plt.show()

KeyboardInterrupt: 

In [9]:
obs.shutdown()